# Research notebook archive

Original project notebook with outputs and personal metadata removed. Existing Colab paths and experiment settings require configuration; see the repository README before running.

In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

In [ ]:
!pip install -U torchao peft transformers accelerate trl

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig, # Import BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)

from trl import SFTTrainer
import string

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
%run /content/drive/MyDrive/NLP_project/code/0_data_wrangling.ipynb

In [ ]:
FEATURE_LABELS = {
    "HHINCOME": "Household income",
    "PERWT": "Person statistical weight",
    "NCHILD": "Number of children",
    "SEX": "Sex",
    "AGE": "Age",
    "AGE_BINS": "Age group",
    "UHRSWORK": "Usual hours worked per week",
    "INCTOT": "Total personal income",
    "INCWAGE": "Wage income",
    "STATE_NAME": "State",
    "OWNERSHP_STATUS": "Home ownership",
    "MARITAL_STATUS": "Marital status",
    "RACE_ETHNICITY": "Race/ethnicity",
    "ENGLISH_STATUS": "English speaking ability",
    "HEALTH_INSURANCE": "Health insurance coverage",
    "EMPLOYMENT_STATUS": "Employment status",
    "EDUCATION_LEVEL": "Education",
    "WORKER_CLASS": "Class of worker",
    "TRANSPORT_METHOD": "Transportation to work",
}
VALUE_MAPS = {
    "STATE_NAME": STATEFIP_map,
    "OWNERSHP_STATUS": OWNERSHP_map,
    "SEX": SEX_map,
    "MARITAL_STATUS": MARST_map,
    "RACE_ETHNICITY": {
        "White": "White",
        "Black/African American": "Black/African American",
        "American Indian or Alaska Native": "American Indian or Alaska Native",
        "Chinese": "Chinese",
        "Japanese": "Japanese",
        "Other Asian or Pacific Islander": "Other Asian or Pacific Islander",
        "Hispanic": "Hispanic",
    },
    "ENGLISH_STATUS": SPEAKENG_map,
    "HEALTH_INSURANCE": HCOVANY_map,
    "EMPLOYMENT_STATUS": EMPSTAT_map,
    "EDUCATION_LEVEL": EDUC_map,
    "WORKER_CLASS": CLASSWKR_map,
    "TRANSPORT_METHOD": TRANWORK_map,
}

LABEL_TO_LONG = {
    'White': 'White',
    'Hispanic': 'Hispanic',
    'Black': 'Black or African American',
    'Indigenous': 'American Indian or Alaska Native',
    'Chinese': 'Chinese',
    'Japanese': 'Japanese',
    'Other': 'Other Asian or Pacific Islander',
}

TARGET_OPTION_ORDER = {
    'RACE_ETHNICITY': ['White', 'Hispanic', 'Black', 'Other', 'Chinese', 'Indigenous', 'Japanese'],
}


In [ ]:
system_prompt = """You are completing one missing demographic attribute for a U.S. survey respondent.

You will be given a profile with one missing attribute and a list of lettered answer options. Predict the most likely value of that missing attribute using the other provided attributes.

Rules:
- Use only the provided profile.
- Choose exactly one option from the list.
- Reply with only the letter of the selected option (A, B, C, ...) — no parentheses, no explanation, no extra text."""


def build_user_prompt(row, target_col, lettered_options):
    """
    lettered_options: list of (letter, display_label) tuples in the order they should appear.
    Example: [("A", "Black or African American"), ("B", "White"), ...]
    """
    profile_lines = build_profile_lines(row, target_col)
    target_label = FEATURE_LABELS[target_col]
    profile_text = "\n".join(profile_lines)
    answers_text = "\n".join([f"({letter}) {label}" for letter, label in lettered_options])

    user_prompt = f"""Profile:

{profile_text}

Missing attribute: {target_label}

Answer options:
{answers_text}

Reply with only the letter of the correct option."""
    return user_prompt

def build_profile_lines(row, target_col):
    lines = []

    for col, label in FEATURE_LABELS.items():
        if col == target_col or col not in row.index or col in ["AGE", "PERWT", "INCTOT"]:
            continue

        value = row[col]
        if pd.isna(value):
            continue

        lines.append(f"- {label}: {value}")

    return lines


# --- Per-row permutation -----------------------------------------------------

def make_letter_map(class_labels, row_seed):
    """
    Produce a reproducible shuffled mapping of letters -> class labels.
    Returns: list of (letter, class_label) in display order.
    """
    rng = np.random.default_rng(row_seed)
    letters = list(string.ascii_uppercase[:len(class_labels)])
    shuffled = list(class_labels)
    rng.shuffle(shuffled)
    return list(zip(letters, shuffled))



In [ ]:
class Stage3Trainer:

    def __init__(
        self,
        feature_train_dfs,
        feature_eval_dfs,
        system_prompt,
        build_user_prompt,
        make_letter_map,
        TARGET_OPTION_ORDER,
        LABEL_TO_LONG,
        model_name="Qwen/Qwen2.5-3B-Instruct",
        seed=42,
    ):

        self.feature_train_dfs = feature_train_dfs
        self.feature_eval_dfs = feature_eval_dfs

        self.system_prompt = system_prompt
        self.build_user_prompt = build_user_prompt
        self.make_letter_map = make_letter_map
        self.TARGET_OPTION_ORDER = TARGET_OPTION_ORDER
        self.LABEL_TO_LONG = LABEL_TO_LONG

        self.model_name = model_name
        self.seed = seed

        self.target = None
        self.classes = None

        self.model = None
        self.tokenizer = None

        self._set_seed()

    # ========================================================
    # SEED
    # ========================================================

    def _set_seed(self):

        random.seed(self.seed)
        np.random.seed(self.seed)
        torch.manual_seed(self.seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.seed)

    # ========================================================
    # TARGET MANAGEMENT
    # ========================================================

    def set_target(self, target):

      self.target = target

      self.train_df = self.feature_train_dfs[target]
      self.eval_df = self.feature_eval_dfs[target]

      self.classes = sorted(
          self.train_df[f"true_{target}"].dropna().unique().tolist()
      )

      print(f"Target set to: {target}")
      print(f"Classes: {self.classes}")
    # ========================================================
    # LOAD BASE MODEL
    # ========================================================

    def load_model(self):

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name
        )

        self.tokenizer.pad_token = self.tokenizer.eos_token

        # Define BitsAndBytesConfig for 4-bit quantization
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=False,
        )

        base_model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            quantization_config=bnb_config, # Use the quantization config
            device_map="auto",
        )

        peft_config = LoraConfig(
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
            ],
        )

        self.model = get_peft_model(base_model, peft_config)

        print("Loaded model + LoRA")

    # ========================================================
    # BUILD SFT DATASET
    # ========================================================

    def build_sft_dataset(self, df):

        rows = []

        for _, row in df.iterrows():

            user_prompt = self.build_user_prompt(
                row=row,
                target_col=self.target,
                lettered_options=self.make_letter_map(self.classes, row["row_seed"])
            )

            answer = row[f"true_{self.target}"]

            # row-specific label map
            label_map = self.make_letter_map(
                self.classes,
                row["row_seed"]
            )

            # Find the correct answer letter
            answer_letter = None

            for letter, label in label_map:
                if label == answer:
                    answer_letter = letter
                    break

            if answer_letter is None:

                raise ValueError(
                    f"True answer '{answer}' not found in label map."
                )

            messages = [
                {
                    "role": "system",
                    "content": self.system_prompt,
                },
                {
                    "role": "user",
                    "content": user_prompt,
                },
                {
                    "role": "assistant",
                    "content": answer_letter,
                },
            ]

            text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )

            rows.append({"text": text})

        return Dataset.from_pandas(pd.DataFrame(rows))

    # ========================================================
    # TRAIN SFT
    # ========================================================

    def train_sft(
        self,
        train_dataset,
        output_dir,
        num_epochs=1,
        batch_size=2,
        grad_accum=8,
        lr=2e-4,
    ):

        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=batch_size,
            gradient_accumulation_steps=grad_accum,
            learning_rate=lr,
            num_train_epochs=num_epochs,
            logging_steps=25,
            save_strategy="no",
                fp16=False,
          bf16=False,
            report_to="none",
        )

        trainer = SFTTrainer(
            model=self.model,
            train_dataset=train_dataset,
            args=training_args,
        )

        trainer.train()

        self.model = trainer.model

        print("Training complete")

    def evaluate_with_logprobs(self, eval_df):

      rows = []

      for _, row in eval_df.iterrows():

          # =====================================================
          # ROW-SPECIFIC LABEL MAP
          # =====================================================

          label_map = self.make_letter_map(
              self.classes,
              row["row_seed"]
          )

          # tuple list -> dict
          reverse_map = {
              letter: label
              for letter, label in label_map
          }

          answer_letters = list(reverse_map.keys())

          answer_token_ids = {
              letter: self.tokenizer.encode(
                  letter,
                  add_special_tokens=False
              )[0]
              for letter in answer_letters
          }

          # =====================================================
          # BUILD PROMPT
          # =====================================================

          user_prompt = self.build_user_prompt(
              row=row,
              target_col=self.target,
              lettered_options=label_map
          )

          messages = [
              {
                  "role": "system",
                  "content": self.system_prompt,
              },
              {
                  "role": "user",
                  "content": user_prompt,
              },
          ]

          text = self.tokenizer.apply_chat_template(
              messages,
              tokenize=False,
              add_generation_prompt=True,
          )

          inputs = self.tokenizer(
              text,
              return_tensors="pt",
          ).to(self.model.device)

          # =====================================================
          # FORWARD PASS
          # =====================================================

          with torch.no_grad():

              outputs = self.model(**inputs)

          logits = outputs.logits[:, -1, :]

          log_probs = torch.log_softmax(
              logits,
              dim=-1,
          )

          # =====================================================
          # STORE ORIGINAL ROW
          # =====================================================

          result = row.to_dict()

          class_probs = {}

          # =====================================================
          # EXTRACT LOGPROBS
          # =====================================================

          for letter, token_id in answer_token_ids.items():

              lp = log_probs[0, token_id].item()

              prob = np.exp(lp)

              class_name = reverse_map[letter]

              result[f"logprob_{class_name}"] = lp

              result[f"prob_{class_name}"] = prob

              class_probs[class_name] = prob

          # =====================================================
          # PREDICTION
          # =====================================================

          pred_class = max(
              class_probs,
              key=class_probs.get
          )

          result[f"pred_{self.target}"] = pred_class

          result["correct"] = (
              pred_class
              ==
              row[f"true_{self.target}"]
          )
          rows.append(result)


      # =========================================================
      # FINAL DATAFRAME
      # =========================================================

      out = pd.DataFrame(rows)

      acc = out["correct"].mean()

      print(f"SFT Accuracy: {acc:.4f}")

      return out


    # ========================================================
    # SAVE MODEL
    # ========================================================

    def save_model(self, save_path, merge=True):

        if merge:

            merged_model = self.model.merge_and_unload()

            merged_model.save_pretrained(save_path)

        else:

            self.model.save_pretrained(save_path)

        self.tokenizer.save_pretrained(save_path)

        print(f"Saved model to: {save_path}")

    # ========================================================
    # LOAD SAVED MODEL
    # ========================================================

    def load_saved_model(self, model_path):

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_path
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            device_map="auto",
        )

        print(f"Loaded saved model from: {model_path}")

In [ ]:
def evaluate_and_plot_comparison(baseline_df, sft_df, target, save=True):
    true_col = f"true_{target}"
    pred_col = f"pred_{target}"

    # ======================
    # 0. FORCE "correct"
    # ======================
    baseline_df = baseline_df.copy()
    sft_df = sft_df.copy()

    baseline_df["correct"] = baseline_df[target] == baseline_df[pred_col]
    sft_df["correct"] = sft_df[true_col] == sft_df[pred_col]

    # ======================
    # ⭐ NEW: TOTAL ACCURACY
    # ======================
    base_total = baseline_df["correct"].mean()
    sft_total = sft_df["correct"].mean()

    print("\n=== Overall Accuracy ===")
    print(f"Baseline: {base_total:.3f}")
    print(f"SFT:      {sft_total:.3f}")
    print(f"Change:   {(sft_total - base_total):.3f}")

    # ======================
    # 1. GROUP SUMMARY
    # ======================
    base_summary = (
        baseline_df
        .groupby(true_col)
        .agg(accuracy=("correct", "mean"), n=("correct", "count"))
        .reset_index()
    )
    base_summary["model_type"] = "baseline"

    sft_summary = (
        sft_df
        .groupby(true_col)
        .agg(accuracy=("correct", "mean"), n=("correct", "count"))
        .reset_index()
    )
    sft_summary["model_type"] = "sft"

    summary = pd.concat([base_summary, sft_summary])

    # ======================
    # 2. PIVOT
    # ======================
    pivot = summary.pivot(
        index=true_col,
        columns="model_type",
        values="accuracy"
    )

    pivot["improvement"] = pivot["sft"] - pivot["baseline"]

    print("\n=== Accuracy Comparison ===")
    print(pivot.round(3))

    # ======================
    # 3. BAR PLOT
    # ======================
    pivot[["baseline", "sft"]].plot(kind="bar")
    plt.title(f"Baseline vs SFT Accuracy ({target})")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=45)
    plt.tight_layout()

    if save:
        plt.savefig(f"{target}_baseline_vs_sft.png")

    plt.show()

    # ======================
    # 4. IMPROVEMENT PLOT
    # ======================
    plt.figure()
    plt.bar(pivot.index, pivot["improvement"])
    plt.axhline(0)
    plt.title(f"SFT Improvement over Baseline ({target})")
    plt.ylabel("Δ Accuracy")
    plt.xticks(rotation=45)
    plt.tight_layout()

    if save:
        plt.savefig(f"{target}_improvement.png")

    plt.show()

    # ======================
    # 5. DISPARITY
    # ======================
    base_disp = base_summary["accuracy"].max() - base_summary["accuracy"].min()
    sft_disp = sft_summary["accuracy"].max() - sft_summary["accuracy"].min()

    print("\n=== Disparity ===")
    print(f"Baseline disparity: {base_disp:.3f}")
    print(f"SFT disparity: {sft_disp:.3f}")
    print(f"Change: {(sft_disp - base_disp):.3f}")

    # ======================
    # 6. WITHIN-PERSON ANALYSIS
    # ======================
    assert "row_id" in baseline_df.columns and "row_id" in sft_df.columns, \
        "Need row_id for per-person comparison"

    merged = baseline_df.merge(
        sft_df,
        on="row_id",
        suffixes=("_baseline", "_sft")
    )

    merged["baseline_correct"] = merged["correct_baseline"]
    merged["sft_correct"] = merged["correct_sft"]

    merged["improved"] = (~merged["baseline_correct"]) & (merged["sft_correct"])
    merged["worsened"] = (merged["baseline_correct"]) & (~merged["sft_correct"])
    merged["unchanged"] = merged["baseline_correct"] == merged["sft_correct"]

    print("\n=== Within-Person Changes ===")
    print(f"Improved:  {merged['improved'].mean():.3f}")
    print(f"Worsened:  {merged['worsened'].mean():.3f}")
    print(f"Unchanged: {merged['unchanged'].mean():.3f}")

    # ======================
    # 7. WITHIN-PERSON BY GROUP
    # ======================
    group_changes = merged.groupby(f"{true_col}_baseline")[["improved", "worsened"]].mean()

    print("\n=== Within-Person Changes by Group ===")
    print(group_changes.round(3))

    # ======================
    # 8. PLOT TRANSITIONS
    # ======================
    plt.figure()
    plt.bar(["Improved", "Worsened", "Unchanged"], [
        merged["improved"].mean(),
        merged["worsened"].mean(),
        merged["unchanged"].mean()
    ])
    plt.title(f"Within-Person Changes ({target})")
    plt.ylabel("Proportion")

    if save:
        plt.savefig(f"{target}_within_person.png")

    plt.show()

    return pivot, merged

# EVALUATING Race, Gender, Age, and Education

Initializing Trainer

In [ ]:
BASE_SAVE_DIR = "/content/drive/MyDrive/NLP_project/data/stage3_outputs/Qwen3-4B-Instruct-2507_05_05V2/"
race_scored = pd.read_parquet("/content/drive/MyDrive/NLP_project/data/stage_1_output_Qwen/Qwen3-4B-Instruct-2507_05_05/phase1_scored_RACE_ETHNICITY.parquet")
education_scored =  pd.read_parquet("/content/drive/MyDrive/NLP_project/data/stage_1_output_Qwen/Qwen3-4B-Instruct-2507_05_05/phase1_scored_EDUCATION_LEVEL.parquet")
age_scored =  pd.read_parquet("/content/drive/MyDrive/NLP_project/data/stage_1_output_Qwen/Qwen3-4B-Instruct-2507_05_05/phase1_scored_AGE_BINS.parquet")
gender_scored =  pd.read_parquet("/content/drive/MyDrive/NLP_project/data/stage_1_output_Qwen/Qwen3-4B-Instruct-2507_05_05/phase1_scored_SEX.parquet")



In [ ]:
from sklearn.model_selection import train_test_split

# ============================================================
# RAW FEATURE DATAFRAMES
# ============================================================

feature_source_dfs = {

   "SEX": gender_scored,

    "AGE_BINS": age_scored,

    "EDUCATION_LEVEL": education_scored,

    "RACE_ETHNICITY": race_scored,
}
TARGETS = list(feature_source_dfs.keys())

# ============================================================
# STORAGE DICTIONARIES
# ============================================================

feature_train_dfs = {}
feature_eval_dfs = {}

# ============================================================
# CREATE SPLITS
# ============================================================

for target, df_target in feature_source_dfs.items():

    print("=" * 60)
    print(target)
    print("=" * 60)

    target_col = f"true_{target}"

    # --------------------------------------------
    # remove missing labels
    # --------------------------------------------

    df_target = df_target.reset_index(drop=True)

    df_target["row_id"] = np.arange(len(df_target))

    # --------------------------------------------
    # stratified split
    # --------------------------------------------

    train_df, eval_df = train_test_split(
        df_target,
        test_size=0.2,
        random_state=42,
        stratify=df_target[target_col],
    )

    # --------------------------------------------
    # reset indices
    # --------------------------------------------

    train_df = train_df.reset_index(drop=True)
    eval_df = eval_df.reset_index(drop=True)

    # --------------------------------------------
    # store
    # --------------------------------------------

    feature_train_dfs[target] = train_df

    feature_eval_dfs[target] = eval_df

    # --------------------------------------------
    # diagnostics
    # --------------------------------------------

    print(f"Train size: {len(train_df)}")
    print(f"Eval size: {len(eval_df)}")

    print()

    print("Train Distribution")

    print(
        train_df[target_col]
        .value_counts(normalize=True)
        .sort_index()
    )

    print()

    print("Eval Distribution")

    print(
        eval_df[target_col]
        .value_counts(normalize=True)
        .sort_index()
    )

    print("\n")

In [ ]:
trainer = Stage3Trainer(
    feature_train_dfs=feature_train_dfs,
    feature_eval_dfs=feature_eval_dfs,
    system_prompt=system_prompt,
    build_user_prompt=build_user_prompt,
    make_letter_map=make_letter_map,
    TARGET_OPTION_ORDER=TARGET_OPTION_ORDER,
    LABEL_TO_LONG=LABEL_TO_LONG,
    model_name = "Qwen/Qwen3-4B-Instruct-2507",
)

trainer.load_model()

all_results = {}


In [ ]:
import os

# ============================================================
# SAVE DIRECTORIES
# ============================================================
os.makedirs(f"{BASE_SAVE_DIR}", exist_ok= True)
os.makedirs(f"{BASE_SAVE_DIR}/results", exist_ok=True)
os.makedirs(f"{BASE_SAVE_DIR}/models", exist_ok=True)
os.makedirs(f"{BASE_SAVE_DIR}/analysis", exist_ok=True)

print("Directories Made")
# ============================================================
# STORAGE
# ============================================================


In [ ]:
import os

# ============================================================
# SAVE DIRECTORIES
# ============================================================
os.makedirs(f"{BASE_SAVE_DIR}", exist_ok= True)
os.makedirs(f"{BASE_SAVE_DIR}/results", exist_ok=True)
os.makedirs(f"{BASE_SAVE_DIR}/models", exist_ok=True)
os.makedirs(f"{BASE_SAVE_DIR}/analysis", exist_ok=True)

print("Directories Made")
# ============================================================
# STORAGE
# ============================================================

all_results = {}

# ============================================================
# TRAINING LOOP
# ============================================================

for step_num, train_target in enumerate(TARGETS):

    print("\n")
    print("#" * 80)
    print(f"STEP {step_num + 1}: TRAIN ON {train_target}")
    print("#" * 80)

    # =====================================================
    # SET TRAIN TARGET
    # =====================================================

    trainer.set_target(train_target)

    # =====================================================
    # BUILD TRAIN DATA
    # =====================================================

    train_ds = trainer.build_sft_dataset(
        trainer.train_df
    )

    # =====================================================
    # TRAIN
    # =====================================================

    trainer.train_sft(
        train_dataset=train_ds,
        output_dir=f"tmp_{train_target}",
        num_epochs=1,
    )

    # =====================================================
    # SAVE CHECKPOINT
    # =====================================================

    checkpoint_path = (
        f"{BASE_SAVE_DIR}/models/"
        f"after_{train_target}"
    )

    trainer.save_model(
        checkpoint_path,
        merge=False,
    )

    print(f"Saved checkpoint to:\n{checkpoint_path}")

    # =====================================================
    # EVALUATE ON TARGET
    # =====================================================
    print("\n")
    print("-" * 60)
    print(f"EVAL TARGET: {train_target}")
    print("-" * 60)

    # =================================================
    # RUN EVALUATION
    # =================================================

    results = trainer.evaluate_with_logprobs(
        trainer.eval_df
    )

    # =================================================
    # STORE RESULTS
    # =================================================

    result_key = (
        f"{train_target}")

    all_results[result_key] = results

    # =================================================
    # SAVE RESULTS
    # =================================================

    results_path = (
        f"{BASE_SAVE_DIR}/results/"
        f"{result_key}_sft.parquet"
    )

    results.to_parquet(results_path)

    print(f"Saved results to:\n{results_path}")

    # =================================================
    # BASELINE DATAFRAME
    # =================================================

    baseline_df = trainer.eval_df

    # =================================================
    # COMPARISON ANALYSIS
    # =================================================

    pivot, merged = evaluate_and_plot_comparison(
        baseline_df=baseline_df,
        sft_df=results,
        target=train_target,
        save=True,
    )

    # =================================================
    # SAVE ANALYSIS TABLES
    # =================================================

    pivot_path = (
        f"{BASE_SAVE_DIR}/analysis/"
        f"{result_key}_pivot.csv"
    )

    merged_path = (
        f"{BASE_SAVE_DIR}/analysis/"
        f"{result_key}_merged.parquet"
    )

    pivot.to_csv(pivot_path)

    merged.to_parquet(merged_path)

    print(f"Saved pivot → {pivot_path}")
    print(f"Saved merged → {merged_path}")

# ============================================================
# SAVE FINAL MERGED MODEL
# ============================================================

final_model_path = (
    f"{BASE_SAVE_DIR}/models/final_merged_model"
)

trainer.save_model(
    final_model_path,
    merge=True,
)

print("\n")
print("=" * 80)
print("FINAL MODEL SAVED")
print("=" * 80)
print(final_model_path)

In [ ]:
final_model_path = (
    f"{BASE_SAVE_DIR}/models/final_adapter_unmerged"
)

trainer.save_model(
    final_model_path,
    merge=False,
)

print("\n")
print("=" * 80)
print("FINAL ADAPTER SAVED")
print("=" * 80)
print(final_model_path)

In [ ]:
display(eval_df)

In [ ]:
display(race_scored)

# Running for Qwen3-4B

In [ ]:
BASE_SAVE_DIR = "/content/drive/MyDrive/NLP_project/data/stage3_outputs/Qwen3-4B/"
race_scored = pd.read_parquet("/content/drive/MyDrive/NLP_project/data/stage_1_output_Qwen/Qwen3-4B/phase1_scored_RACE_ETHNICITY.parquet")
education_scored =  pd.read_parquet("/content/drive/MyDrive/NLP_project/data/stage_1_output_Qwen/Qwen3-4B/phase1_scored_EDUCATION_LEVEL.parquet")
age_scored =  pd.read_parquet("/content/drive/MyDrive/NLP_project/data/stage_1_output_Qwen/Qwen3-4B/phase1_scored_AGE_BINS.parquet")
gender_scored =  pd.read_parquet("/content/drive/MyDrive/NLP_project/data/stage_1_output_Qwen/Qwen3-4B/phase1_scored_SEX.parquet")



In [ ]:
from sklearn.model_selection import train_test_split

# ============================================================
# RAW FEATURE DATAFRAMES
# ============================================================

feature_source_dfs = {

   "SEX": gender_scored,

    "AGE_BINS": age_scored,

    "EDUCATION_LEVEL": education_scored,

    "RACE_ETHNICITY": race_scored,
}
TARGETS = list(feature_source_dfs.keys())

# ============================================================
# STORAGE DICTIONARIES
# ============================================================

feature_train_dfs = {}
feature_eval_dfs = {}

# ============================================================
# CREATE SPLITS
# ============================================================

for target, df_target in feature_source_dfs.items():

    print("=" * 60)
    print(target)
    print("=" * 60)

    target_col = f"true_{target}"

    # --------------------------------------------
    # remove missing labels
    # --------------------------------------------

    df_target = df_target.reset_index(drop=True)

    df_target["row_id"] = np.arange(len(df_target))

    # --------------------------------------------
    # stratified split
    # --------------------------------------------

    train_df, eval_df = train_test_split(
        df_target,
        test_size=0.2,
        random_state=42,
        stratify=df_target[target_col],
    )

    # --------------------------------------------
    # reset indices
    # --------------------------------------------

    train_df = train_df.reset_index(drop=True)
    eval_df = eval_df.reset_index(drop=True)

    # --------------------------------------------
    # store
    # --------------------------------------------

    feature_train_dfs[target] = train_df

    feature_eval_dfs[target] = eval_df

    # --------------------------------------------
    # diagnostics
    # --------------------------------------------

    print(f"Train size: {len(train_df)}")
    print(f"Eval size: {len(eval_df)}")

    print()

    print("Train Distribution")

    print(
        train_df[target_col]
        .value_counts(normalize=True)
        .sort_index()
    )

    print()

    print("Eval Distribution")

    print(
        eval_df[target_col]
        .value_counts(normalize=True)
        .sort_index()
    )

    print("\n")

In [ ]:
trainer = Stage3Trainer(
    feature_train_dfs=feature_train_dfs,
    feature_eval_dfs=feature_eval_dfs,
    system_prompt=system_prompt,
    build_user_prompt=build_user_prompt,
    make_letter_map=make_letter_map,
    TARGET_OPTION_ORDER=TARGET_OPTION_ORDER,
    LABEL_TO_LONG=LABEL_TO_LONG,
    model_name = "Qwen/Qwen3-4B-Instruct-2507",
)

trainer.load_model()

all_results = {}


In [ ]:
import os

# ============================================================
# SAVE DIRECTORIES
# ============================================================

BASE_SAVE_DIR = "/content/drive/MyDrive/NLP_Project/data/stage3_outputs"

os.makedirs(f"{BASE_SAVE_DIR}/results", exist_ok=True)
os.makedirs(f"{BASE_SAVE_DIR}/models", exist_ok=True)
os.makedirs(f"{BASE_SAVE_DIR}/analysis", exist_ok=True)

# ============================================================
# STORAGE
# ============================================================

all_results = {}

# ============================================================
# TRAINING LOOP
# ============================================================

for step_num, train_target in enumerate(TARGETS):

    print("\n")
    print("#" * 80)
    print(f"STEP {step_num + 1}: TRAIN ON {train_target}")
    print("#" * 80)

    # =====================================================
    # SET TRAIN TARGET
    # =====================================================

    trainer.set_target(train_target)

    # =====================================================
    # BUILD TRAIN DATA
    # =====================================================

    train_ds = trainer.build_sft_dataset(
        trainer.train_df
    )

    # =====================================================
    # TRAIN
    # =====================================================

    trainer.train_sft(
        train_dataset=train_ds,
        output_dir=f"tmp_{train_target}",
        num_epochs=1,
    )

    # =====================================================
    # SAVE CHECKPOINT
    # =====================================================

    checkpoint_path = (
        f"{BASE_SAVE_DIR}/models/"
        f"after_{train_target}"
    )

    trainer.save_model(
        checkpoint_path,
        merge=False,
    )

    print(f"Saved checkpoint to:\n{checkpoint_path}")

    # =====================================================
    # EVALUATE ON TARGET
    # =====================================================
    print("\n")
    print("-" * 60)
    print(f"EVAL TARGET: {train_target}")
    print("-" * 60)

    # =================================================
    # RUN EVALUATION
    # =================================================

    results = trainer.evaluate_with_logprobs(
        trainer.eval_df
    )

    # =================================================
    # STORE RESULTS
    # =================================================

    result_key = (
        f"{train_target}")

    all_results[result_key] = results

    # =================================================
    # SAVE RESULTS
    # =================================================

    results_path = (
        f"{BASE_SAVE_DIR}/results/"
        f"{result_key}_sft.parquet"
    )

    results.to_parquet(results_path)

    print(f"Saved results to:\n{results_path}")

    # =================================================
    # BASELINE DATAFRAME
    # =================================================

    baseline_df = trainer.eval_df

    # =================================================
    # COMPARISON ANALYSIS
    # =================================================

    pivot, merged = evaluate_and_plot_comparison(
        baseline_df=baseline_df,
        sft_df=results,
        target=train_target,
        save=True,
    )

    # =================================================
    # SAVE ANALYSIS TABLES
    # =================================================

    pivot_path = (
        f"{BASE_SAVE_DIR}/analysis/"
        f"{result_key}_pivot.csv"
    )

    merged_path = (
        f"{BASE_SAVE_DIR}/analysis/"
        f"{result_key}_merged.parquet"
    )

    pivot.to_csv(pivot_path)

    merged.to_parquet(merged_path)

    print(f"Saved pivot → {pivot_path}")
    print(f"Saved merged → {merged_path}")

# ============================================================
# SAVE FINAL MERGED MODEL
# ============================================================

final_model_path = (
    f"{BASE_SAVE_DIR}/models/final_merged_model"
)

trainer.save_model(
    final_model_path,
    merge=True,
)

print("\n")
print("=" * 80)
print("FINAL MODEL SAVED")
print("=" * 80)
print(final_model_path)

In [ ]:
final_model_path = (
    f"{BASE_SAVE_DIR}/models/final_adapter_unmerged"
)

trainer.save_model(
    final_model_path,
    merge=False,
)

print("\n")
print("=" * 80)
print("FINAL ADAPTER SAVED")
print("=" * 80)
print(final_model_path)